In [1]:
# Устанавливаем необходимые библиотеки (только если их нет)
!pip install -q transformers datasets peft accelerate evaluate trl bitsandbytes triton

# Импортируем библиотеки
import torch
import numpy as np
import time
import gc
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
from huggingface_hub import login
from google.colab import userdata
import evaluate
from tqdm import tqdm

# Аутентификация на Hugging Face (токен хранится в секретах Colab)
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.6 MB/s eta 0:00:00


In [2]:
# Загружаем датасет
dataset = load_dataset("phatvo/hotpotqa-raft-1k", split="train")
print("Колонки датасета:", dataset.column_names)
# Вывод: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction', 'text']

# Посмотрим первый пример, чтобы понять структуру
print(dataset[0])

README.md:   0%|          | 0.00/649 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Колонки датасета: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction', 'text']
{'id': 'seed_task_0', 'type': 'general', 'question': 'When was the song "All Is Full of Love" fully released as a single?', 'context': {'sentences': [['The Amazing Adventures of Kavalier & Clay is a 2000 novel by Jewish American author Michael Chabon that won the Pulitzer Prize for Fiction in 2001. The novel follows the lives of two Jewish cousins, Czech artist Joe Kavalier and Brooklyn-born writer Sammy Clay, before, during, and after World War II. In the novel, Kavalier and Clay become major figures in the comics industry from its nascency into its Golden Age. "Kavalier & Clay" was published to "nearly unanimous praise" and became a "New York Times" Best Seller, receiving nominations for the 2000 National Book Critics Circle Award and PEN/Faulkner Award for Fiction. In 2006, Bret Easton Ellis declared the novel "one of the three great books of my generation," and in 2007, "T

In [3]:
# Фиксируем seed для воспроизводимости
np.random.seed(42)
indices = np.random.permutation(len(dataset))
train_indices = indices[:80]   # первые 80 для обучения
test_indices = indices[80:100] # следующие 20 для тестирования

train_dataset = dataset.select(train_indices)
test_dataset = dataset.select(test_indices)

print(f"Размер обучающей выборки: {len(train_dataset)}")
print(f"Размер тестовой выборки: {len(test_dataset)}")

Размер обучающей выборки: 80
Размер тестовой выборки: 20


In [4]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Устанавливаем pad_token = eos_token, чтобы избежать предупреждений
tokenizer.pad_token = tokenizer.eos_token

# Загружаем модель в bfloat16 для экономии памяти
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Проверяем, что модель загрузилась на GPU
print("Устройство:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Устройство: cuda:0


In [5]:
# Конфигурация LoRA: rank=8, alpha=32, dropout=0.1
# Целевые модули для Qwen (q_proj, k_proj, v_proj, o_proj)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# Применяем LoRA к модели
# Примечание: предупреждение про отсутствующий triton можно игнорировать,
# оно влияет только на максимальную скорость вычислений, но не на логику работы.
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Выведет количество обучаемых параметров (в миллионах)

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [6]:
def format_training_example(example):
    """Форматирует пример для обучения: user = Context + Question, assistant = cot_answer"""
    user_content = f"Context:\n{example['context']}\n\nQuestion: {example['question']}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example['cot_answer']}
    ]
    # add_generation_prompt=False, так как мы включаем ответ ассистента
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": prompt}

def format_inference_example(example):
    """Форматирует пример для инференса: только user, с generation prompt"""
    user_content = f"Context:\n{example['context']}\n\nQuestion: {example['question']}"
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt}

# Применяем форматирование
train_dataset = train_dataset.map(format_training_example)
test_dataset_raw = test_dataset.map(format_inference_example)  # для инференса
# Для оценки перплексии нам нужен будет тестовый датасет в формате как для обучения
test_dataset_eval = test_dataset.map(format_training_example)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [7]:
def tokenize_function(examples):
    """Токенизирует текст и создает метки (labels = input_ids)"""
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,          # может потребоваться увеличить, если контексты длинные
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Токенизируем обучающий и оценочный (для перплексии) датасеты
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
test_tokenized = test_dataset_eval.map(tokenize_function, batched=True, remove_columns=test_dataset_eval.column_names)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [9]:
# Аргументы обучения
training_args = TrainingArguments(
    output_dir="./qwen-raft-lora",
    per_device_train_batch_size=2,      # небольшой batch для Colab
    per_device_eval_batch_size=2,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",               # будем оценивать на test после каждой эпохи
    eval_steps=100,
    save_total_limit=1,
    learning_rate=2e-5,
    bf16=True,                           # используем bfloat16
    fp16=False,
    push_to_hub=False,
    report_to="none",
    gradient_accumulation_steps=2,       # имитируем batch=4
)

# Коллатор данных (языковое моделирование)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Инициализируем Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,        # будем вычислять loss на тесте во время обучения
    data_collator=data_collator,
    processing_class=tokenizer,         # обновлено: в новых версиях используется processing_class вместо tokenizer
)

In [10]:
# Очищаем кэш GPU перед началом
torch.cuda.reset_peak_memory_stats()

# Засекаем время
start_time = time.time()

# Запускаем обучение
trainer.train()

# Время окончания
end_time = time.time()
training_time_min = (end_time - start_time) / 60
print(f"Время обучения: {training_time_min:.2f} мин")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss


Время обучения: 0.37 мин


In [11]:
# Получаем максимальную выделенную память в байтах и переводим в ГБ
peak_memory_bytes = torch.cuda.max_memory_allocated()
peak_memory_gb = peak_memory_bytes / (1024 ** 3)
print(f"Пиковое использование GPU памяти: {peak_memory_gb:.2f} ГБ")

Пиковое использование GPU памяти: 7.13 ГБ


In [12]:
trainable_params = model.num_parameters(only_trainable=True) / 1e6
print(f"Обучаемые параметры: {trainable_params:.2f} млн")

Обучаемые параметры: 1.08 млн


In [13]:
# После обучения вычисляем loss на тестовом датасете
eval_results = trainer.evaluate(test_tokenized)
eval_loss = eval_results["eval_loss"]
perplexity = np.exp(eval_loss)
print(f"Perplexity на тесте: {perplexity:.4f}")

Perplexity на тесте: 9.4157


In [14]:
# Переводим модель в режим оценки
model.eval()

def compute_f1(pred, truth):
    """Вычисляет F1 на уровне слов (разбиение по пробелам)"""
    pred_tokens = pred.split()
    truth_tokens = truth.split()
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1

predictions = []
references = []

# Генерируем ответы для каждого тестового примера
for example in tqdm(test_dataset_raw, desc="Генерация ответов"):
    input_ids = tokenizer.encode(example["prompt"], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=128,
            do_sample=False,              # жадная декодировка
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(generated)
    references.append(example["cot_answer"])   # используем cot_answer как эталон

# Вычисляем F1 и Exact Match
f1_scores = []
exact_matches = []
for pred, ref in zip(predictions, references):
    f1 = compute_f1(pred, ref)
    f1_scores.append(f1)
    exact_matches.append(1 if pred.strip() == ref.strip() else 0)

avg_f1 = np.mean(f1_scores)
em = np.mean(exact_matches)
print(f"Средний F1: {avg_f1:.4f}")
print(f"Exact Match: {em:.4f}")

Генерация ответов:   0%|          | 0/20 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Генерация ответов: 100%|██████████| 20/20 [00:47<00:00,  2.36s/it]

Средний F1: 0.1729
Exact Match: 0.0000


In [15]:
print("\n=== Результаты эксперимента ===")
print(f"Модель: {model_name} + LoRA")
print(f"Обучаемые параметры: {trainable_params:.2f} млн")
print(f"Время обучения: {training_time_min:.2f} мин")
print(f"Пиковое использование GPU памяти: {peak_memory_gb:.2f} ГБ")
print(f"Perplexity на тесте: {perplexity:.4f}")
print(f"F1 на тесте: {avg_f1:.4f}")


=== Результаты эксперимента ===
Модель: Qwen/Qwen2.5-0.5B-Instruct + LoRA
Обучаемые параметры: 1.08 млн
Время обучения: 0.37 мин
Пиковое использование GPU памяти: 7.13 ГБ
Perplexity на тесте: 9.4157
F1 на тесте: 0.1729
